In [1]:
!module list
!nvidia-smi
!which mpirun
!which python3
%env HOROVOD_GPU_ALLREDUCE=NCCL
%env HOROVOD_GPU_ALLGATHER=NCCL
%env HOROVOD_GPU_BROADCAST=NCCL
%env NCCL_DEBUG=DEBUG

Currently Loaded Modulefiles:
 1) intel-mkl/2020.3.304   5) nccl/2.10.3-cuda11.4   9) conda/DL_hvd_0221  
 2) python3/3.9.2          6) openmpi/4.1.1         10) pbs                
 3) cuda/11.4.1            7) tensorflow/2.6.0      
 4) cudnn/8.2.2-cuda11.4   8) horovod/0.22.1        
>Tue Sep  1 19:48:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.173.02             Driver Version: 580.173.02     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                

In [2]:
!nvidia-smi -L

GPU 0: NVIDIA H200 (UUID: GPU-037f1eb4-825a-8ed6-c3e3-1d6a816588cc)


In [3]:
import horovod
#from horovod.run.run import run
from horovod import run
def dummy():
    #----------------------------------------
    # Import packages
    #----------------------------------------
    import sys
    import os
    os.environ["TF_DISABLE_NVTX_RANGES"] = "1"
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
    os.environ["NCCL_DEBUG"] = "WARN"
    import time
    import socket
    import math
    import numpy as np
    import xarray as xr
    import dask
    import dask.array as da
    import zarr as zr
    import pickle
    import tensorflow as tf
    print(tf.version)
    from tensorflow import keras
    import horovod.tensorflow.keras as hvd
    #import horovod.tensorflow as hvd
    from tensorflow.keras import layers
    from tensorflow.keras import models, losses
    from tensorflow.keras.regularizers import l1,l2
    from tensorflow.keras.optimizers import Optimizer
    # Set the logging level to suppress warnings
    tf.get_logger().setLevel(tf.compat.v1.logging.ERROR)
    #------------------------------------------------------------
    # Intialise Horovod
    # -----------------------------------------------------------
    hvd.init()
    print ('***hvd.size ', hvd.size(),' hvd.rank', hvd.rank(), 'hvd.local_rank() ', hvd.local_rank())
    # Horovod: pin GPU to be used to process local rank (one GPU per process)

    print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
    gpus = tf.config.experimental.list_physical_devices('GPU')
    print(' gpus = ', gpus)
    if hvd.local_rank() == 0:
        print("Socket and len gpus = ",socket.gethostname(), len(gpus))
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    if gpus:
        tf.config.experimental.set_visible_devices(gpus[hvd.local_rank()], 'GPU')

        
    #----------------------------------------------------------------------------------------------
    #
    # Build downscaling model using TensorFlow and Horovod - this version uses convolutuions
    #
    #----------------------------------------------------------------------------------------------

    def SRDCNN_SST_v3(numHiddenUnits, numResponses, numFeatures, numLats, numLongs, shrink):

        concat_axis = -1
        activation = 'relu'
        shrink = shrink
        reg_val  = 0.000000001

        inputs  = tf.keras.layers.Input(shape = (int(numLats/shrink), int(numLongs/shrink), numFeatures) )

        # Downscale model using conv2dTranspose layers

        x = layers.Conv2DTranspose(numHiddenUnits, (7, 7), strides=2, activation="relu", padding="same",
                    kernel_regularizer=l2(reg_val), activity_regularizer=l2(reg_val), bias_regularizer=l2(reg_val))(inputs)

        x = layers.Conv2DTranspose(numHiddenUnits, (7, 7), strides=2, activation="relu", padding="same",
                    kernel_regularizer=l2(reg_val), activity_regularizer=l2(reg_val), bias_regularizer=l2(reg_val))(x)

        x = layers.Conv2DTranspose(numHiddenUnits, (7, 7), strides=2, activation="relu", padding="same",
                    kernel_regularizer=l2(reg_val), activity_regularizer=l2(reg_val), bias_regularizer=l2(reg_val))(x)

        x = layers.Conv2D(numResponses, (1, 1), activation="linear", padding="same",
            kernel_regularizer=l2(reg_val), activity_regularizer=l2(reg_val), bias_regularizer=l2(reg_val))(x) 

        model = tf.keras.models.Model(inputs=inputs, outputs=x)

    # Horovod: adjust learning rate based on number of GPUs.
        opt = tf.optimizers.Adam(lr=0.003, beta_1=0.9, beta_2=0.999,
                   epsilon=None, decay=0.0, amsgrad=False)
        opt = hvd.DistributedOptimizer(opt)

        model.compile(loss='mse', optimizer=opt, metrics=['msle','mae'], experimental_run_tf_function=False)

        print( model.summary() ) if hvd.rank() == 0 else None

        return model

    # 

    t0 = [0]*hvd.size()
    t0[hvd.rank()] = time.time()

    # Set total number of images, training epoch size, test data size
    #
    #  - must be Scalable/divisible to multi nodes to ensure load balance

    # 36 years of daily data [1979-2014]📍️

    Total_images = 13149 #for SST daily data
    Epoch_size = 10560 #approx. 80%
    Test_size  = Total_images - Epoch_size #2560 if needs to be divisible by 80

    # Batch size - aim to fill GPU memory to achieve best computational performance

    batch_size = 16

    # Set key parameters for Conv2DLSTM

    numHiddenUnits = 64
    numResponses = 1
    numFeatures  = 1
    shrink = 8

    numLats        = 512 
    numLongs       = 512

    if hvd.rank() == 0:
        print ('*** rank = ', hvd.rank(),' Epoch size = ', Epoch_size)
        print ('*** rank = ', hvd.rank(),' Test_size = ', Test_size)
        print ('*** rank = ', hvd.rank(),' Batch size = ', batch_size)
        print ('*** rank = ', hvd.rank(),' numHiddenUnits = ', numHiddenUnits)
        print ('*** rank = ', hvd.rank(),' numResponses= ', numResponses)
        print ('*** rank = ', hvd.rank(),' numFeatures = ', numFeatures)
        print ('*** rank = ', hvd.rank(),' numLats = ', numLats)
        print ('*** rank = ', hvd.rank(),' numLongs = ', numLongs)
    #----------------------------------------------------------------------------------------------
    #
    # 	Open the input data files using xarray
    #
    #----------------------------------------------------------------------------------------------

    # #   NCI

    ###------------------ Use the standardised data
    folder = "/g/data/sd82/"
    ds = xr.open_dataset(folder + "sst_stand_10km_OFAM_historical_Australia_lon_interp.nc")
    
    sst = ds["temp"]
    
    # Remove depth dimension if it is length 1
    if "st_ocean" in sst.dims and sst.sizes["st_ocean"] == 1:
        sst = sst.squeeze("st_ocean")
        
    print("Shape after squeeze:", sst.shape)
    
    if "lon_sst" in ds:
        lon = ds["lon_sst"]
        lat = ds["lat_sst"]
    elif "lon" in ds:
        lon = ds["lon"]
        lat = ds["lat"]
    elif "xt_ocean" in ds:
        lon = ds["xt_ocean"]
        lat = ds["yt_ocean"]
    else:
        raise ValueError("No longitude/latitude variables found")

    time_past = ds.Time.data
    sst_data = sst.data #saved standardised data
    
    print("CNN input shape:", sst_data.shape)

    #-----------------------------------------
    # Compute and print file open time
    elapsed_time = time.time() - t0[hvd.rank()]
    print ('*** rank = ', hvd.rank(),' Dataset Intitialise Elapsed Time (sec) = ', elapsed_time)

    #----------------------------------------------------------------------------------------------
    #
    # Horovod: Split the test data across multiple processors   
    #
    #----------------------------------------------------------------------------------------------

    istart = int(hvd.rank()*Epoch_size/hvd.size())
    istop  = int((hvd.rank()+1)*Epoch_size/hvd.size())

    i_test_start = int(hvd.rank()*Test_size/hvd.size()+Epoch_size) + 1
    i_test_stop  = int((hvd.rank()+1)*Test_size/hvd.size()+Epoch_size)

    if i_test_stop >= Total_images:
        i_test_stop = Total_images - 1

    print ( '*** rank = ', hvd.rank(),' istart = ', istart, ' istop = ', istop)
    print ( '*** rank = ', hvd.rank(),' i_test_start = ', i_test_start, ' i_test_stop = ', i_test_stop)

    t0[hvd.rank()] = time.time()
    #----------------------------------------------------------------------------------------------
    #
    # Horovod: Read in the train and test data across multiple processors
    #
    #----------------------------------------------------------------------------------------------

    #  Downscale experiments start with low resolution data


    #add a tensor to make the data 4D
    x_tr_n = np.expand_dims(sst_data[istart:istop,:,:], axis=3)
    x_te_n = np.expand_dims(sst_data[i_test_start:i_test_stop,:,:], axis=3)

    x_train = tf.keras.layers.AveragePooling2D(
              pool_size=(shrink, shrink), strides=None, padding='same', data_format=None)(x_tr_n)
    x_test = tf.keras.layers.AveragePooling2D(
              pool_size=(shrink, shrink), strides=None, padding='same', data_format=None)(x_te_n)



    #  Target is the high resolution data

    y_train = x_tr_n
    y_test = x_te_n

    print(' Training data shapes = ',x_train.shape, y_train.shape)
    print(' Test data shapes = ',x_test.shape, y_test.shape)

    print ('*** rank = ', hvd.rank(),' Y data')

    print ('*** rank = ', hvd.rank(),' y_train', sys.getsizeof(y_train))
    print ('*** rank = ', hvd.rank(),' y_test', sys.getsizeof(y_test))


    print ('*** rank = ', hvd.rank(),' X data')

    print ('*** rank = ', hvd.rank(),' x_train', sys.getsizeof(x_train))
    print ('*** rank = ', hvd.rank(),' x_test', sys.getsizeof(x_test))

    #   Elapsed time for read operation

    elapsed_time = time.time() - t0[hvd.rank()]
    print ('*** rank = ', hvd.rank(),' Dataset Read Elapsed Time (sec) = ', elapsed_time)

    # Determine how many batches are there in train and test sets

    train_batches = len(x_train) // batch_size
    test_batches = len(x_test) // batch_size

    print ('*** rank = ', hvd.rank(),' train_batches', train_batches)
    print ('*** rank = ', hvd.rank(),' test_batches', test_batches)
    #----------------------------------------------------------------------------------------------
    #
    # Horovod: create callbacks required for horovod model run
    #
    #----------------------------------------------------------------------------------------------

    callbacks = [
        # Horovod: broadcast initial variable states from rank 0 to all other processes.
        # This is necessary to ensure consistent initialization of all workers when
        # training is started with random weights or restored from a checkpoint.
        hvd.callbacks.BroadcastGlobalVariablesCallback(0),

        # Horovod: average metrics among workers at the end of every epoch.
        #
        # Note: This callback must be in the list before the ReduceLROnPlateau,
        # TensorBoard or other metrics-based callbacks.
        hvd.callbacks.MetricAverageCallback(),]
    #----------------------------------------------------------------------------------------------
    #
    # Horovod: save checkpoints only on worker 0 to prevent other workers from corrupting them.
    #
    #----------------------------------------------------------------------------------------------

    if hvd.rank() == 0:
        callbacks.append(tf.keras.callbacks.ModelCheckpoint('./checkpoints_200/checkpoint-{epoch}.h5', monitor='val_loss', save_best_only=True))

    #----------------------------------------------------------------------------------------------
    #
    # 	Build the model
    #
    #----------------------------------------------------------------------------------------------

    model = SRDCNN_SST_v3(numHiddenUnits, numResponses, numFeatures, numLats, numLongs, shrink)

    #----------------------------------------------------------------------------------------------
    #
    # 	Train the model
    #
    #----------------------------------------------------------------------------------------------

    # Setup timer for training step

    t0[hvd.rank()] = time.time()

    # Add a barrier to sync all processes before starting training

    hvd.allreduce([0], name="Barrier")
    print ('*** rank = ', hvd.rank(),' Train model')

    history = model.fit(x_train, y_train, callbacks=callbacks, epochs=50, verbose=2, 
                          validation_data = (x_test, y_test))
    if hvd.rank() == 0:
        print(history.history)

    # Elapsed time for training operation
    elapsed_time = time.time() - t0[hvd.rank()]
    print ('*** rank = ', hvd.rank(),' Total Training Elapsed Time (sec) = ', elapsed_time)

In [4]:
%%time
#if __name__ == '__main__':
run(dummy, use_mpi=True, np=4)

[1,1]<stdout>:<module 'tensorflow._api.v2.version' from '/apps/tensorflow/2.6.0/lib/python3.9/site-packages/tensorflow/_api/v2/version/__init__.py'>
[1,3]<stdout>:<module 'tensorflow._api.v2.version' from '/apps/tensorflow/2.6.0/lib/python3.9/site-packages/tensorflow/_api/v2/version/__init__.py'>
[1,0]<stdout>:<module 'tensorflow._api.v2.version' from '/apps/tensorflow/2.6.0/lib/python3.9/site-packages/tensorflow/_api/v2/version/__init__.py'>
[1,2]<stdout>:<module 'tensorflow._api.v2.version' from '/apps/tensorflow/2.6.0/lib/python3.9/site-packages/tensorflow/_api/v2/version/__init__.py'>
[1,2]<stdout>:***hvd.size  4  hvd.rank 2 hvd.local_rank()  2
[1,0]<stdout>:***hvd.size  4  hvd.rank 0 hvd.local_rank()  0
[1,3]<stdout>:***hvd.size  4  hvd.rank 3 hvd.local_rank()  3
[1,1]<stdout>:***hvd.size  4  hvd.rank 1 hvd.local_rank()  1
[1,2]<stdout>:Num GPUs Available:  1


[1,2]<stderr>:User function raise error: list index out of rangeTraceback (most recent call last):


[1,2]<stdout>: gpus =  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


[1,2]<stderr>:  File "/g/data/xv83/cxs599/conda/envs/DL_hvd_0221/lib/python3.9/runpy.py", line 197, in _run_module_as_main


[1,1]<stdout>:Num GPUs Available:  1


...

[1,0]<stdout>:*** rank =  0  numLongs =  512


[1,2]<stderr>:    main(driver_addr, run_func_server_port)
[1,2]<stderr>:  File "/apps/horovod/0.22.1/tensorflow/lib/python3.9/site-packages/horovod-0.22.1-py3.9-linux-x86_64.egg/horovod/runner/run_task.py", line 28, in main
[1,1]<stderr>:    main(driver_addr, run_func_server_port)
[1,1]<stderr>:  File "/apps/horovod/0.22.1/tensorflow/lib/python3.9/site-packages/horovod-0.22.1-py3.9-linux-x86_64.egg/horovod/runner/run_task.py", line 28, in main
[1,2]<stderr>:    raise e
[1,2]<stderr>:  File "/apps/horovod/0.22.1/tensorflow/lib/python3.9/site-packages/horovod-0.22.1-py3.9-linux-x86_64.egg/horovod/runner/run_task.py", line 25, in main
[1,1]<stderr>:    raise e
[1,1]<stderr>:  File "/apps/horovod/0.22.1/tensorflow/lib/python3.9/site-packages/horovod-0.22.1-py3.9-linux-x86_64.egg/horovod/runner/run_task.py", line 25, in main
[1,2]<stderr>:    ret_val = func()
[1,2]<stderr>:  File "/apps/horovod/0.22.1/tensorflow/lib/python3.9/site-packages/horovod-0.22.1-py3.9-linux-x86_64.egg/horovod/runne

[1,0]<stdout>:Shape after squeeze: (13149, 512, 512)


--------------------------------------------------------------------------
Primary job  terminated normally, but 1 process returned
a non-zero exit code. Per user-direction, the job has been aborted.
--------------------------------------------------------------------------
--------------------------------------------------------------------------
mpirun detected that one or more processes exited with non-zero status, thus causing
the job to be terminated. The first process to do so was:

  Process name: [[7301,1],3]
  Exit code:    1
--------------------------------------------------------------------------


RuntimeError: mpirun failed with exit code 1

In [6]:
import xarray as xr

ds = xr.open_dataset("/g/data/sd82/sst_stand_10km_OFAM_historical_Australia_lon_interp.nc")

print(ds)
print("\nVariables:")
print(list(ds.variables))


<xarray.Dataset>
Dimensions:    (Time: 13149, st_ocean: 1, yt_ocean: 512, xt_ocean: 512)
Coordinates:
  * Time       (Time) datetime64[ns] 1979-01-01T12:00:00 ... 2014-12-31T12:00:00
  * st_ocean   (st_ocean) float64 2.5
  * xt_ocean   (xt_ocean) float64 107.3 107.4 107.6 107.7 ... 158.2 158.4 158.4
  * yt_ocean   (yt_ocean) float64 -52.95 -52.85 -52.75 ... -2.05 -1.95 -1.85
Data variables:
    temp       (Time, st_ocean, yt_ocean, xt_ocean) float32 ...
    temp_mean  float32 ...
    temp_std   float32 ...
Attributes:
    filename:       TMP/ocean_ofam_1979_01.nc.0000
    NumFilesInSet:  720
    title:          jra_55_1979
    grid_type:      regular
    history:        Fri May 29 16:42:18 2026: ncks -O -d xt_ocean,43,554 sst_...
    NCO:            netCDF Operators version 5.0.5 (Homepage = http://nco.sf....

Variables:
['Time', 'st_ocean', 'temp', 'xt_ocean', 'yt_ocean', 'temp_mean', 'temp_std']
